# 02b — SQL Analysis
**Project:** Customer Lifecycle & Campaign Effectiveness Analysis  
**Dataset:** Online Retail II (UCI Machine Learning Repository)  
**Author:** Amrit Sharma  

---

## Objective
Run analytical SQL queries against a local SQLite database built from the outputs of Notebooks 01 and 02.  
Two tables: `transactions` (779,425 rows) and `rfm_segments` (5,878 customers including churn predictions from Notebook 03).

Queries cover segment revenue analysis, customer lifetime value, monthly trends, channel behaviour, and churn prediction cross-referencing.

---

## 0. Setup & Imports

In [ ]:
import os
import sqlite3
import pandas as pd

# ── Path configuration ────────────────────────────────────────────────────────
BASE_DIR = os.path.expanduser('~/Desktop/Customer Lifecycle Analysis Project')

print(f'Project root : {BASE_DIR}')
print(f'Data folder  : {os.path.join(BASE_DIR, "data")}')
print(f'Folders exist: data={os.path.exists(os.path.join(BASE_DIR, "data"))}, outputs={os.path.exists(os.path.join(BASE_DIR, "outputs"))}')
print('Libraries loaded ✓')

## 1. Build the Database

Loading both CSV outputs into a local SQLite database.  
SQLite runs entirely inside Python — no server or installation required.  
The `.db` file is created in the `/data/` folder alongside the CSVs.

In [ ]:
# ── Connect to database ───────────────────────────────────────────────────────
DB_PATH = os.path.join(BASE_DIR, 'data', 'customer_lifecycle.db')
conn    = sqlite3.connect(DB_PATH)

# ── Load CSVs ─────────────────────────────────────────────────────────────────
rfm = pd.read_csv(os.path.join(BASE_DIR, 'data', 'rfm_with_churn.csv'), dtype={'Customer ID': str})
df  = pd.read_csv(os.path.join(BASE_DIR, 'data', 'online_retail_clean.csv'),
                  dtype={'Customer ID': str}, parse_dates=['InvoiceDate'])

# ── Write to database ─────────────────────────────────────────────────────────
rfm.to_sql('rfm_segments', conn, if_exists='replace', index=False)
df.to_sql('transactions',  conn, if_exists='replace', index=False)

print(f'Database     : {DB_PATH}')
print(f'rfm_segments : {len(rfm):,} rows')
print(f'transactions : {len(df):,} rows')
print(f'Total rows   : {len(rfm) + len(df):,}')

## 2. Query Helper Function

A reusable function that takes a SQL string, runs it against the database, and prints clean results.  
Calling it with a semicolon `;` suppresses the duplicate DataFrame display in Jupyter.

In [ ]:
def run_query(sql, description=''):
    result = pd.read_sql_query(sql, conn)
    if description:
        print(f'── {description} ──')
    print(result.to_string(index=False))
    print(f'\n{len(result)} rows returned\n')
    return result

print('Helper function ready ✓')

## 3. Queries

### Query 1 — Segment Revenue Summary
GROUP BY with multiple aggregations. Baseline summary confirming segment sizes and revenue from the RFM analysis.

In [ ]:
run_query("""
    SELECT 
        Segment,
        COUNT(*) as customer_count,
        ROUND(SUM(Monetary), 2) as total_revenue,
        ROUND(AVG(Monetary), 2) as avg_revenue
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY total_revenue DESC
""", 'Query 1 — Segment Revenue Summary');

### Query 2 — Revenue Concentration (Window Function)
Uses `SUM() OVER ()` — a window function that calculates the grand total across all rows simultaneously, enabling percentage calculation without a subquery.

In [ ]:
run_query("""
    SELECT 
        Segment,
        ROUND(SUM(Monetary), 2) as revenue,
        ROUND(SUM(Monetary) * 100.0 / SUM(SUM(Monetary)) OVER (), 1) as revenue_pct
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY revenue DESC
""", 'Query 2 — Revenue Concentration by Segment');

### Query 3 — Top 10 Customers by Lifetime Value
Simple SELECT with ORDER BY and LIMIT. Identifies the highest-value individual customers.

In [ ]:
run_query("""
    SELECT 
        "Customer ID",
        Recency,
        Frequency,
        ROUND(Monetary, 2) AS Monetary,
        Segment
    FROM rfm_segments
    ORDER BY Monetary DESC
    LIMIT 10
""", 'Query 3 — Top 10 Customers by Lifetime Value');

### Query 4 — Monthly Revenue Trend (JOIN)
INNER JOIN between `transactions` and `rfm_segments` on Customer ID. Confirms the Q4 seasonality pattern found in Notebook 01 using SQL.

In [ ]:
run_query("""
    SELECT 
        t.InvoiceMonth,
        COUNT(DISTINCT t.Invoice) AS orders,
        ROUND(SUM(t.TotalRevenue), 2) AS revenue
    FROM transactions t
    JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
    GROUP BY t.InvoiceMonth
    ORDER BY t.InvoiceMonth
""", 'Query 4 — Monthly Revenue Trend');

### Query 5 — Champion Purchasing Patterns (JOIN + WHERE)
Filters the JOIN to Champions only. Shows average unit price, quantity, and product diversity for the highest-value segment.

In [ ]:
run_query("""
    SELECT 
        r.Segment,
        ROUND(AVG(t.Price), 2) AS avg_unit_price,
        ROUND(AVG(t.Quantity), 2) AS avg_quantity,
        COUNT(DISTINCT t.StockCode) AS unique_products_bought
    FROM transactions t
    JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
    WHERE r.Segment = 'Champions'
    GROUP BY r.Segment
""", 'Query 5 — Champion Purchasing Patterns');

### Query 6 — High Value At Risk Customers (Multiple WHERE Conditions)
Filters for At Risk customers with Monetary > £1,000. Produces the manual CRM override list — customers the model underscores but who represent significant historical value.

In [ ]:
run_query("""
    SELECT
        "Customer ID",
        Recency,
        Frequency,
        ROUND(Monetary, 2) AS Monetary
    FROM rfm_segments
    WHERE Segment = 'At Risk'
      AND Monetary > 1000
    ORDER BY Monetary DESC
""", 'Query 6 — High Value At Risk Customers');

### Query 7 — Country & Segment Breakdown (Multi-column GROUP BY + HAVING)
Groups by two columns simultaneously. HAVING filters out country-segment combinations with fewer than 10 customers to keep results meaningful.

In [ ]:
run_query("""
    SELECT 
        t.Country,
        r.Segment,
        COUNT(DISTINCT r."Customer ID") as customers,
        ROUND(SUM(t.TotalRevenue), 2) as revenue
    FROM transactions t
    JOIN rfm_segments r ON t."Customer ID" = r."Customer ID"
    GROUP BY t.Country, r.Segment
    HAVING COUNT(DISTINCT r."Customer ID") > 10
    ORDER BY revenue DESC
""", 'Query 7 — Country & Segment Breakdown');

### Query 8 — Predicted Churners by Segment
Cross-references churn predictions from Notebook 03 with RFM segments. Confirms the model results in SQL — Lost and Potential Loyal at highest predicted churn, Champions at lowest.

In [ ]:
run_query("""
    SELECT 
        Segment,
        COUNT(*) as total_customers,
        SUM(Churn_Predicted) as predicted_churners,
        ROUND(AVG(Churn_Probability) * 100, 1) as avg_churn_probability_pct
    FROM rfm_segments
    GROUP BY Segment
    ORDER BY avg_churn_probability_pct DESC
""", 'Query 8 — Predicted Churners by Segment');

## 4. Close Connection

In [ ]:
conn.close()
print('Database connection closed ✓')

## Summary

Eight SQL queries run against a local SQLite database built from the cleaned
transaction and RFM segment data produced in Notebooks 01 and 02.

Queries range from simple aggregations to multi-table JOINs, window functions
for revenue concentration, and cross-referencing churn predictions from
Notebook 03 against segment data.

Key finding: UK Champions (1,353 customers) generate £9.7M — over half of
total revenue from a single country-segment combination. Lost customers
represent 25.9% of the customer base but only 3.8% of revenue, confirming
the business case for differentiated CRM strategy built across this project.